# Baseline Regression Models

This notebook trains baseline regression models to predict continuous JNK3 and GSK3β docking scores using the selected RDKit molecular descriptors. Ridge Regression, Random Forest and HistGradientBoosting models are trained on the official training split and evaluated on the validation split using MAE, RMSE, R² and Spearman correlation. The held-out scaffold test set is not used during this baseline model comparison.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import spearmanr


DATA_FILE = "preprocessed_model_data.csv"
DESCRIPTOR_FILE = "selected_descriptor_names.csv"
RANDOM_SEED = 42


data_df = pd.read_csv(DATA_FILE)

descriptor_columns = pd.read_csv(
    DESCRIPTOR_FILE
)["descriptor"].tolist()


print("Dataset shape:", data_df.shape)
print("Number of descriptors:", len(descriptor_columns))

In [ ]:
# Prepare the official training and validation data

train_df = data_df[
    data_df["split"] == "train"
].copy()

validation_df = data_df[
    data_df["split"] == "validation"
].copy()


X_train = train_df[descriptor_columns]
X_validation = validation_df[descriptor_columns]


# GSK3B docking-score targets
y_gsk3b_train = train_df["gsk3b_score"]
y_gsk3b_validation = validation_df["gsk3b_score"]


# JNK3 docking-score targets
y_jnk3_train = train_df["jnk3_score"]
y_jnk3_validation = validation_df["jnk3_score"]


print("Training molecules:", len(train_df))
print("Validation molecules:", len(validation_df))

print("\nTraining feature shape:", X_train.shape)
print("Validation feature shape:", X_validation.shape)

print("\nThe test set has not been used.")

In [ ]:
#  Create fresh regression models and one evaluation function

def create_regression_models():

    return {
        "Ridge Regression": Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "regressor",
                Ridge(alpha=1.0)
            )
        ]),

        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ),

        "Gradient Boosting": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            random_state=RANDOM_SEED
        )
    }


def calculate_regression_metrics(
    target_name,
    model_name,
    observed_scores,
    predicted_scores
):

    mae = mean_absolute_error(
        observed_scores,
        predicted_scores
    )

    rmse = np.sqrt(
        mean_squared_error(
            observed_scores,
            predicted_scores
        )
    )

    r2 = r2_score(
        observed_scores,
        predicted_scores
    )

    spearman_correlation = spearmanr(
        observed_scores,
        predicted_scores
    ).statistic

    return {
        "target": target_name,
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "spearman_correlation": spearman_correlation
    }


print("Regression models and evaluation function are ready.")

In [ ]:
# Train all regression models

targets = {
    "GSK3B": (
        y_gsk3b_train,
        y_gsk3b_validation
    ),

    "JNK3": (
        y_jnk3_train,
        y_jnk3_validation
    )
}


regression_results = []
prediction_tables = []


for target_name, target_values in targets.items():

    y_train_target, y_validation_target = target_values

    # Create fresh models for this target
    regression_models = create_regression_models()

    for model_name, model in regression_models.items():

        print(
            "Training",
            model_name,
            "for",
            target_name
        )

        # Train the model
        model.fit(
            X_train,
            y_train_target
        )

        # Predict validation docking scores
        predicted_scores = model.predict(
            X_validation
        )

        # Calculate model performance
        result = calculate_regression_metrics(
            target_name,
            model_name,
            y_validation_target,
            predicted_scores
        )

        regression_results.append(result)

        # Save the predictions in a table
        predictions = pd.DataFrame({
            "canonical_smiles": validation_df[
                "canonical_smiles"
            ].values,

            "target": target_name,
            "model": model_name,
            "observed_score": y_validation_target.values,
            "predicted_score": predicted_scores
        })

        prediction_tables.append(predictions)

        # Save the trained model
        safe_target = target_name.lower()

        safe_model = (
            model_name
            .lower()
            .replace(" ", "_")
        )

        joblib.dump(
            model,
            f"{safe_target}_{safe_model}_regression_model.joblib"
        )

        print(model_name, "completed.\n")


print("All regression models trained successfully.")

In [ ]:
# Compare and save the regression results

regression_results_df = pd.DataFrame(
    regression_results
).sort_values(
    by=["target", "mae"]
)

regression_predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True
)


display(
    regression_results_df.round(3)
)


regression_results_df.to_csv(
    "regression_validation_results.csv",
    index=False
)

regression_predictions_df.to_csv(
    "regression_validation_predictions.csv",
    index=False
)


print("Saved: regression_validation_results.csv")
print("Saved: regression_validation_predictions.csv")